# BERT Context

Compare token boundaries and contextual similarities. Does high cosine imply task accuracy?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/google-bert/bert-base-uncased

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    BertTokenizer, BertModel, BertForMaskedLM,
    CLIPProcessor, CLIPModel, CLIPTokenizerFast,
)
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
BERT_ID = 'bert-base-uncased'

print(f"📦 Loading BERT: {BERT_ID}")
bert_tokenizer = BertTokenizer.from_pretrained(BERT_ID)
bert_model = BertModel.from_pretrained(BERT_ID).to(device)
bert_model.eval()

bert_mlm = BertForMaskedLM.from_pretrained(BERT_ID).to(device)
bert_mlm.eval()

params = sum(p.numel() for p in bert_model.parameters())
print(f"   ✅ Loaded!")
print(f"   - Parameters: {params / 1e6:.1f}M")
print(f"   - Hidden size: {bert_model.config.hidden_size}")
print(f"   - Layers: {bert_model.config.num_hidden_layers}")
print(f"   - Attention heads: {bert_model.config.num_attention_heads}")
print(f"   - Vocab size: {bert_tokenizer.vocab_size}")

In [ ]:
@torch.no_grad()
def get_bert_embeddings(texts, pooling='mean'):
    """
    Extract sentence embeddings with BERT.

    Args:
        pooling: 'cls' = [CLS] token, 'mean' = average over all tokens
    """
    encoded = bert_tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors='pt'
    ).to(device)

    outputs = bert_model(**encoded)

    if pooling == 'cls':
        # Output of the [CLS] token
        embeddings = outputs.last_hidden_state[:, 0, :]
    else:
        # Mean pooling (respecting the attention mask)
        attention_mask = encoded['attention_mask'].unsqueeze(-1)
        token_embeds = outputs.last_hidden_state
        embeddings = (token_embeds * attention_mask).sum(1) / attention_mask.sum(1)

    # L2 normalize
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().numpy()

print("✅ get_bert_embeddings function defined")

## Demo code

Compare token boundaries and contextual similarities. Does high cosine imply task accuracy?

In [ ]:
sentences = ["A puppy is running.", "A dog is playing.", "The market closed."]
vectors = get_bert_embeddings(sentences)
similarity = vectors @ vectors.T
print(np.round(similarity, 3))

In [ ]:
# Show the tokenization process step by step
sentences = [
    "A golden retriever playing in the park.",
    "The cat is sleeping on the sofa.",
    "Dogs and cats are popular pets worldwide.",
]

print("📝 BERT tokenization:\n")
for sent in sentences:
    tokens = bert_tokenizer.tokenize(sent)
    token_ids = bert_tokenizer.encode(sent)

    print(f"  Original:  {sent}")
    print(f"  Tokens:    {tokens}")
    print(f"  Token IDs: {token_ids}")
    print(f"  Decoded:   {bert_tokenizer.decode(token_ids)}")
    print(f"  Token #:   {len(tokens)}")
    print()

In [ ]:
# Description of special tokens
example = "The dog is cute."
encoded = bert_tokenizer(example, return_tensors='pt')

print("🔤 BERT Special Tokens:")
print(f"  [CLS] = {bert_tokenizer.cls_token} (ID: {bert_tokenizer.cls_token_id}) → token representing the whole sentence")
print(f"  [SEP] = {bert_tokenizer.sep_token} (ID: {bert_tokenizer.sep_token_id}) → sentence separator")
print(f"  [MASK] = {bert_tokenizer.mask_token} (ID: {bert_tokenizer.mask_token_id}) → masked token used in MLM")
print(f"  [PAD] = {bert_tokenizer.pad_token} (ID: {bert_tokenizer.pad_token_id}) → padding")
print()

tokens = bert_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
print(f"  Input: '{example}'")
print(f"  Tokens: {tokens}")
print(f"         ↑[CLS]                               ↑[SEP]")

In [ ]:
# Sentence similarity experiment
test_sentences = [
    # Dog-related
    "A golden retriever playing fetch in the park.",
    "The puppy is wagging its tail happily.",
    "Dogs are loyal and friendly companions.",
    # Cat-related
    "A tabby cat sleeping on the windowsill.",
    "The kitten is chasing a ball of yarn.",
    "Cats are independent and graceful animals.",
    # Unrelated sentences
    "The stock market crashed yesterday.",
    "Python is a popular programming language.",
]

embeddings = get_bert_embeddings(test_sentences)
sim_matrix = cosine_similarity(embeddings)

# Visualization
fig, ax = plt.subplots(figsize=(12, 10))

short_labels = [
    "Dog: retriever playing", "Dog: puppy wagging", "Dog: loyal companions",
    "Cat: tabby sleeping", "Cat: kitten chasing", "Cat: independent",
    "Finance: stock market", "Tech: python language"
]

im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-0.2, vmax=1)
ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_title('BERT Sentence Similarity (Cosine)', fontsize=14, fontweight='bold')

for i in range(len(sim_matrix)):
    for j in range(len(sim_matrix)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8,
               color='white' if abs(sim_matrix[i,j]) > 0.6 else 'black')

# Group divider lines
for pos in [2.5, 5.5]:
    ax.axhline(y=pos, color='black', linewidth=2)
    ax.axvline(x=pos, color='black', linewidth=2)

plt.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

print("💡 Observation: Dog sentences are similar to each other; cat sentences are similar to each other.")
print("   Similarity to the finance/tech sentences is low.")

In [ ]:
# Dog/Cat-related text descriptions
dog_texts = [
    "A photo of a dog.", "A cute puppy running.",
    "Golden retriever in the yard.", "A dog fetching a ball.",
    "A friendly dog wagging its tail.", "Labrador swimming in water.",
    "A corgi with short legs.", "A husky in the snow.",
    "Dogs are loyal animals.", "A beagle sniffing the ground.",
]

cat_texts = [
    "A photo of a cat.", "A cute kitten playing.",
    "Tabby cat on the sofa.", "A cat chasing a mouse.",
    "A cat purring softly.", "Persian cat with long fur.",
    "A black cat at night.", "Siamese cat with blue eyes.",
    "Cats are independent animals.", "A cat grooming itself.",
]

all_texts = dog_texts + cat_texts
text_labels = ['dog'] * len(dog_texts) + ['cat'] * len(cat_texts)

# Extract BERT embeddings
bert_text_embeds = get_bert_embeddings(all_texts, pooling='mean')
print(f"✅ BERT text embeddings: {bert_text_embeds.shape}")

# t-SNE
tsne = TSNE(n_components=2, perplexity=8, random_state=SEED)
bert_2d = tsne.fit_transform(bert_text_embeds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) t-SNE
for cls, color, marker in [('dog', '#4ECDC4', 'o'), ('cat', '#FF6B6B', 's')]:
    mask = np.array(text_labels) == cls
    axes[0].scatter(bert_2d[mask, 0], bert_2d[mask, 1], c=color, marker=marker,
                   s=100, alpha=0.8, edgecolors='white', label=cls.upper())
    for i, (x, y) in enumerate(bert_2d[mask]):
        idx = np.where(mask)[0][i]
        axes[0].annotate(all_texts[idx][:25] + '...', (x, y), fontsize=7,
                        textcoords='offset points', xytext=(5, 5), alpha=0.7)

axes[0].set_title('BERT Text Embeddings (t-SNE)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(alpha=0.2)

# (2) Cosine Similarity
sim = cosine_similarity(bert_text_embeds)
im = axes[1].imshow(sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].axhline(y=len(dog_texts)-0.5, color='black', linewidth=2)
axes[1].axvline(x=len(dog_texts)-0.5, color='black', linewidth=2)
axes[1].set_title('Text Cosine Similarity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Dog texts ← | → Cat texts')
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## Interpretation

Unfine-tuned BERT pooling is not a calibrated sentence similarity model.

Compare the output with the evidence in the input. A generated answer can be wrong.